# Quantify the difference in gap sizes between cells in two images
## Offers the option to quantify absolute values or compare the % difference between two images

### Intended for quantifying the ability of a given segmentation algorithm to fill in gaps between cells 

In [36]:
import numpy as np
from skimage import measure

#get gap regions
def find_gaps(img):
    """
    Find regions of background that are completely enclosed by non-zero pixels i.e. gaps between cells. Any regions of background that are touching the edges of the image are ignored, as these are considered large chunks of background
    
    """
    #Create binary mask of background pixels
    background_mask = (img == 0)

    #label connected background regions
    labelled = measure.label(background_mask, connectivity = 2)
    regions = measure.regionprops(labelled)

    gaps = []
    h, w = img.shape[:2]

    for region in regions:
        coords = region.coords 

        #check if any pixel is touching an image border
        touches_border = (
            np.any(coords[:, 0] == 0) or      # top edge
            np.any(coords[:, 0] == h - 1) or  # bottom edge
            np.any(coords[:, 1] == 0) or      # left edge
            np.any(coords[:, 1] == w - 1)     # right edge
        )

        if not touches_border:
            gaps.append(region)

    return gaps

def find_absolute_area(img):
    gaps = find_gaps(img)
    
    areas = []
    for gap in gaps:
        areas.append(gap.area)

    absolute_area = sum(areas)
    return absolute_area


def gap_percentage_change(img1, img2):
    """ Find the value of Img2's gaps as a percentage of Img1 gaps. 
    """

    area1 = find_absolute_area(img1)
    area2 = find_absolute_area(img2)

    percentage_change = area2/area1 * 100
    if percentage_change == float("inf"): 
        print("image1 has no gaps, percentage change cannot be calculated")

    else:
        return percentage_change

# Usage

In [ ]:
import tifffile as tf

#read in images
img1 = tf.imread("/path/to/img1")
img2 = tf.imread("/path/to/img2")

print(find_absolute_area(img1))
print(find_absolute_area(img1))

gap_percentage_change(img1, img2)


9942.0
9942.0


np.float64(68.32629249647958)